# Production Notebook — Student Success Prediction
## DS602 Final Project — Spring 2026

This notebook is intentionally separate from the project notebook. The whole point of keeping them apart is that on presentation day, the professor provides a brand new dataset — one we've never seen — and this notebook should run on it cleanly without any modifications.

All the heavy lifting (clustering, SME queries, model training, hyperparameter tuning) was done in the project notebook. This notebook just loads what was saved and applies it to new data. Think of it as the deployment step.

**How to use it:**
- The professor will provide two URLs: one for the unlabeled features (`X_path`) and one for the true labels (`y_path`)
- Swap those into the `production()` call at the bottom of this notebook
- Run all cells top to bottom
- The classification report will print automatically

---
## Step 1: Imports

We only import what we actually need here — no clustering libraries, no cross-validation utilities, nothing from the training phase. The production notebook is intentionally lightweight. If you find yourself adding sklearn clustering imports here, something has gone wrong with the design.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report

---
## Step 2: Feature Engineering Function

This function is a direct copy of the one in the project notebook — same formulas, same column names, same order. That's not an accident. The model was trained on features produced by this exact function, so if we change anything here (even something small like the weights in `engagement_score`), the predictions will be wrong in ways that are hard to debug.

If you ever update the feature engineering in the project notebook, this function must be updated to match. They have to stay in sync.

In [ ]:
def engineer_features(df_input):
    """Exact same feature engineering pipeline used during training.
    Must match the project notebook identically — same formulas, same column names.
    """
    d = df_input.copy()

    d['attendance_rate'] = (
        d['classes_attended'] / d['classes_total'].replace(0, np.nan)
    ).fillna(0).clip(0, 1)

    d['assignment_completion_rate'] = (
        d['assignments_submitted'] / d['assignments_total'].replace(0, np.nan)
    ).fillna(0).clip(0, 1)

    d['quiz_score_rate'] = (
        d['quiz_points_earned'] / d['quiz_points_possible'].replace(0, np.nan)
    ).fillna(0).clip(0, 1)

    d['practice_exam_rate'] = (
        d['practice_exam_points'] / d['practice_exam_possible'].replace(0, np.nan)
    ).fillna(0).clip(0, 1)

    d['late_submission_ratio'] = (
        d['late_submissions'] / d['assignments_submitted'].replace(0, np.nan)
    ).fillna(0).clip(0, 1)

    d['engagement_score'] = (
        d['attendance_rate'] * 0.3 +
        d['assignment_completion_rate'] * 0.3 +
        (d['discussion_posts'] / (d['discussion_posts'].max() + 1e-9)) * 0.2 +
        (d['lms_logins_last_30_days'] / (d['lms_logins_last_30_days'].max() + 1e-9)) * 0.2
    )

    d['total_burden'] = d['work_hours_per_week'] + (d['commute_minutes'] / 60)

    d['academic_performance'] = (
        d['quiz_score_rate'] * 0.5 +
        d['practice_exam_rate'] * 0.3 +
        (d['prior_gpa'] / 4.0) * 0.2
    )

    d['workload_index'] = d['total_burden'] / (d['study_hours_per_week'] + 1)

    return d

---
## Step 3: Production Function and Evaluation

The `production()` function ties everything together. Here's what happens when you call it:

1. **Load the saved model** — we load `student_success_model.pkl`, which is the full sklearn Pipeline (scaler + best classifier) saved at the end of the project notebook. Because it's a Pipeline, it handles all preprocessing automatically — no need to manually scale the new data.

2. **Load the feature list** — `model_features.pkl` tells us exactly which columns the model expects, and in what order. This prevents mismatches if the new dataset happens to have columns in a different order.

3. **Apply feature engineering** — we run the new data through `engineer_features()` to create the same derived features the model was trained on.

4. **Generate predictions** — the model predicts `0` (at-risk) or `1` (likely to succeed) for every student in the new dataset.

5. **Evaluate against true labels** — we load the labels file the professor provides and print a full classification report: precision, recall, F1, and support for both classes.

The last two lines call the function with the professor's dataset URLs. On presentation day, just swap in the new URLs if they're different.

In [6]:
def production(X_path, y_path):
    # Load trained pipeline and feature list
    model         = joblib.load('student_success_model.pkl')
    MODEL_FEATURES = joblib.load('model_features.pkl')

    # Load new dataset
    df_X = pd.read_csv(X_path)

    # Apply identical feature engineering as training
    df_X = engineer_features(df_X)

    # Predict
    pred = model.predict(df_X[MODEL_FEATURES])

    # Load true labels and evaluate
    df_Y    = pd.read_csv(y_path)
    y_label = df_Y.columns[-1]
    df_y    = df_Y[y_label]

    print(classification_report(df_y, pred, target_names=['At-Risk (0)', 'Success (1)']))


production(
    y_path='https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/prod-student_success_labels.csv',
    X_path='https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/prod-student_success_unlabeled.csv'
)

              precision    recall  f1-score   support

 At-Risk (0)       0.96      0.63      0.76      1069
 Success (1)       0.78      0.98      0.87      1431

    accuracy                           0.83      2500
   macro avg       0.87      0.81      0.81      2500
weighted avg       0.86      0.83      0.82      2500

